# 🔐 Chapter 1: The Machine

## *How the Enigma worked — and how we simulate it in Python*

---

### The Atlantic, 1941

U-boats were slaughtering Allied convoys. Every evening, weather reports,
tactical orders, and fleet positions were radioed across the North Atlantic —
all encrypted with a device the German military believed was unbreakable.

Intercept stations in Britain were listening. Thousands of messages piled up
at Bletchley Park. They were meaningless: strings of letters that looked like
random noise.

The device responsible was the **Enigma machine**.

---

### What Was Enigma?

Enigma was an electro-mechanical cipher machine. An operator pressed a key
on the keyboard; a different letter lit up on the lamp board — the ciphertext
letter. A second operator wrote down what lit up. The result was a message
that (supposedly) no one without the day's settings could read.

What made it clever was a set of **rotating wired wheels** — the *rotors* —
that changed the substitution with *every single keypress*. Press A twice in a
row and you'd get two completely different output letters. This ruled out the
simple frequency analysis that had broken every earlier cipher.

---

### The Components

```
KEYBOARD  →  PLUGBOARD  →  ROTOR R  →  ROTOR M  →  ROTOR L  →  REFLECTOR
                                                                      │
LAMPBOARD ←  PLUGBOARD  ←  ROTOR R  ←  ROTOR M  ←  ROTOR L  ←──────┘
```

| Component | German name | What it does |
|---|---|---|
| Rotors (×3) | Walzen | Wired scrambler wheels that rotate with each keypress |
| Reflector | Umkehrwalze (UKW) | Sends the signal back through the rotors |
| Plugboard | Steckerbrett | Swaps pairs of letters before and after the rotors |

Let's build each one.


## Part 1: The Rotor

A rotor is a disc with 26 electrical contacts on each face. Internal wiring
connects each contact on the right face to a *different* contact on the left
face — a fixed scrambled substitution.

But here's the key: the rotor **rotates**. After each keypress, the rightmost
rotor advances one position. When it completes a full revolution, it steps the
middle rotor. And so on — like an odometer.

The wiring for the historical Wehrmacht Rotor I (the one most often used) is:

```
Input:   A B C D E F G H I J K L M N O P Q R S T U V W X Y Z
Output:  E K M F L G D Q V Z N T O W Y H X U S P A I B R C J
```

So if a signal enters at contact A (position 0), it exits at E (position 4).


In [ ]:
import sys
sys.path.insert(0, '..')

from src.enigma_bayes.enigma.rotor import Rotor, ROTOR_SPECS, ALPHABET

# Inspect the historical rotor wirings
for name in ['I', 'II', 'III']:
    spec = ROTOR_SPECS[name]
    print(f'Rotor {name}: wiring={spec.wiring}  turnover={spec.turnovers!r}')

In [ ]:
# Create Rotor I at position A (0), ring setting A (0)
rotor_I = Rotor(ROTOR_SPECS['I'], ring_setting=0, position=0)

# Pass signal 0 (= letter A) through it in the forward direction
output = rotor_I.forward(0)
print(f'A → {ALPHABET[output]}  (expected: E)')  # Rotor I: A→E

# And the backward (inverse) direction
output_back = rotor_I.backward(output)
print(f'E → {ALPHABET[output_back]} (round-trip, expected: A)')

### The Ring Setting (Ringstellung)

Each rotor also has a **ring setting**: a number (A–Z) that offsets the
wiring relative to the alphabet ring on the outside of the wheel.

Think of it as rotating the *internal* wiring without rotating the *outer*
alphabet ring. It changes every possible substitution the rotor makes.

This was set once per day as part of the daily key settings.


In [ ]:
# Demonstrate the effect of ring settings
for ring in [0, 1, 5]:
    r = Rotor(ROTOR_SPECS['I'], ring_setting=ring, position=0)
    out = r.forward(0)  # what does A map to?
    ring_letter = ALPHABET[ring]
    print(f'Ring setting {ring_letter}: A → {ALPHABET[out]}')

## Part 2: The Reflector

After the signal travels right-to-left through all three rotors, it hits
the **reflector** — a fixed wiring disc at the far left of the machine.

The reflector routes the signal *back* through the rotors from left to right.

Critically, the reflector's wiring is **self-inverse**: if A maps to Y,
then Y maps to A. This means Enigma encryption and decryption are the same
operation — the same machine and settings decrypt as encrypt.

But it has an unavoidable consequence: **a letter can never encrypt to itself**.
This seems like a minor quirk. Turing would turn it into a wrecking ball.


In [ ]:
from src.enigma_bayes.enigma.reflector import Reflector, REFLECTOR_SPECS

ref = Reflector(REFLECTOR_SPECS['UKW-B'])

# Verify it is self-inverse
print('Verifying UKW-B is self-inverse:')
all_ok = True
for i in range(26):
    if ref.reflect(ref.reflect(i)) != i:
        all_ok = False
        print(f'  ERROR at position {i}')
        break
if all_ok:
    print('  ✓ Every letter maps back to itself after two reflections.')

# Demonstrate the no-self-encryption property
print('\nReflector maps:')
for i in range(5):
    j = ref.reflect(i)
    print(f'  {ALPHABET[i]} → {ALPHABET[j]} (and {ALPHABET[j]} → {ALPHABET[i]})')

## Part 3: The Plugboard

The plugboard (Steckerbrett) sat at the front of the machine, a patch panel
of sockets connected by cables. The operator plugged up to 10 cables in,
each swapping a pair of letters *before* they entered the rotors and *after*
they returned.

A 10-cable plugboard configuration multiplies the keyspace by roughly
150 trillion. It was — correctly — considered the most significant
contribution to Enigma's security.


In [ ]:
from src.enigma_bayes.enigma.plugboard import Plugboard

# Create a plugboard with 3 pairs
pb = Plugboard(['AB', 'QZ', 'RT'])
print(pb)

# A gets swapped to B and back
print(f'\nPlugboard swap: A → {pb.swap_letter("A")} (expect B)')
print(f'Plugboard swap: B → {pb.swap_letter("B")} (expect A)')
print(f'Plugboard swap: C → {pb.swap_letter("C")} (expect C, unplugged)')

## Part 4: Assembling the Full Machine

Now let's put it all together. The `EnigmaMachine` class wires up the
plugboard, three rotors, and the reflector into a working simulation.

A complete machine configuration requires:
- Which **three rotors** to use, and in which order
- The **starting position** of each rotor (the *Grundstellung*)
- The **ring setting** of each rotor (the *Ringstellung*)
- Which **reflector** to use
- Which **plugboard pairs** to connect

All of these changed daily, specified in a key sheet distributed in advance
to all units. Capturing one of those key sheets was one of the most prized
intelligence coups of the war.


In [ ]:
from src.enigma_bayes.enigma.machine import EnigmaMachine, EnigmaConfig

# Reconstruct the settings from a famous historical message.
# On 1 November 1942, U-559 was sunk and its Enigma materials captured.
# Here we use a simplified example with known settings.

config = EnigmaConfig.from_letters(
    rotors=('I', 'II', 'III'),
    positions=('A', 'A', 'A'),
    ring_settings=('A', 'A', 'A'),
    reflector='UKW-B',
    plugboard_pairs=[],  # no plugboard for clarity
)

machine = EnigmaMachine(config)
plaintext  = 'HELLOTURINGSWORLD'
ciphertext = machine.encrypt(plaintext)
print(f'Plaintext:  {plaintext}')
print(f'Ciphertext: {ciphertext}')

In [ ]:
# KEY DEMO: same settings decrypt back to plaintext
# (because the reflector makes Enigma self-inverse)
machine.reset(config)  # reset to starting position
decrypted = machine.encrypt(ciphertext)
print(f'Decrypted:  {decrypted}')
assert decrypted == plaintext, 'Something went wrong!'
print('✓ Encryption and decryption are identical operations on Enigma.')

In [ ]:
# Observe rotor stepping in action
machine.reset(config)
print('Rotor window positions while encrypting AAAAAAAAAA:')
print(f'  Start: {machine.window_positions()}')
for i, letter in enumerate('AAAAAAAAAA'):
    enc = machine.encrypt_letter(letter)
    pos = machine.window_positions()
    print(f'  Keypress {i+1}: A → {enc}   window: {pos}')

## Part 5: The Keyspace — Why Brute Force Is Impossible

How many possible Enigma settings are there? Let's count.


In [ ]:
from math import factorial, comb

# Rotor selection and order: choose 3 from 5, in order
rotor_orderings = factorial(5) // factorial(5 - 3)  # = P(5,3) = 60

# Rotor starting positions: 26^3
rotor_positions = 26 ** 3

# Ring settings: 26^3 (but the third one is redundant — 26^2 effective)
ring_settings = 26 ** 2  # simplified

# Plugboard: choose 10 pairs from 26 letters
# C(26,2) × C(24,2) × ... / 10! = number of ways to choose 10 unordered pairs
plugboard_configs = 1
available = 26
for _ in range(10):
    plugboard_configs *= comb(available, 2)
    available -= 2
plugboard_configs //= factorial(10)  # pairs are unordered

total = rotor_orderings * rotor_positions * ring_settings * plugboard_configs

print(f'Rotor orderings (choose 3 of 5): {rotor_orderings:>20,}')
print(f'Rotor starting positions (26^3):  {rotor_positions:>20,}')
print(f'Ring settings (26^2, simplified): {ring_settings:>20,}')
print(f'Plugboard configurations (10 of 26): {plugboard_configs:>17,}')
print(f'─' * 50)
print(f'Total possible settings:          {total:>20,}')
print()

# At 1 billion tests per second
seconds = total / 1e9
years   = seconds / (365.25 * 24 * 3600)
print(f'At 10^9 tests/second: {years:.2e} years to brute-force.')
print('(The universe is ~13.8 × 10^9 years old.)')

## Summary

We've built a working Enigma machine from first principles:

- **`Rotor`**: implements the wired substitution with position and ring offset
- **`Reflector`**: self-inverse wiring that routes the signal back
- **`Plugboard`**: symmetric letter-pair swapper
- **`EnigmaMachine`**: assembles all components with correct stepping logic

The keyspace is astronomical — **~1.07 × 10²³ possible settings** — making
brute force completely infeasible even today.

And yet, Turing's team broke it routinely.

**In Chapter 2**, we'll see the mathematical tool that made it possible:
a theorem published in 1763 by Thomas Bayes that lets you update beliefs
with evidence — and a crucial flaw in Enigma's design that gave those
beliefs somewhere to start.

> *"The impossibility of the task seemed to make it more attractive."*
> — Alan Turing, on arriving at Bletchley Park
